In [42]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub


In [43]:
from langchain.agents import create_react_agent, AgentExecutor


In [44]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

In [45]:
search_tool = TavilySearchResults(
    max_results=2,
)

In [46]:
@tool
def get_weather_data(city: str) -> str:
    # Placeholder function to simulate fetching weather data
    """Fetch weather data for a given city."""

    url=(
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHER_API_KEY}&query={city}"
    )
    response = requests.get(url)
    data = response.json()
    if "current" not in data:
        return f"Could not retrieve weather data for {city}."
    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%\n"
    )

In [20]:
result=search_tool.invoke("latest news on AI research")
result

[{'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': "Date: June 18, 2026\n\nSummary: Researchers at the University of Kansas developed PP-VAE, an AI model that improves electrocardiogram analysis while protecting patient privacy by reducing exposure of biometric information like age and sex. The system uses advanced neural architectures to separate clinically relevant signals from identifiable personal characteristics while maintaining diagnostic accuracy for heart disease and mortality risk prediction. Published in Scientific Reports, the model demonstrates competitive performance with existing approaches while strengthening privacy safeguards. Future plans include testing across diverse datasets and public release to enable secure cross-institutional health data sharing.\n\nSource: Digital Watch Observatory [...] Date: June 17, 2026\n\nSummary: Researchers led by Professor Han Zhang at Shenzhen University developed an all-fiber photonic AI platform using

In [7]:
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
# 3. Iterate through the server's live registry
print("Available Gemini Models for your API Key:\n" + "-"*40)
for model in genai.list_models():
    # Filter for models that specifically support text generation
    if 'generateContent' in model.supported_generation_methods:
        print(f"Model String: {model.name}")
        print(f"Description: {model.description}\n")

Available Gemini Models for your API Key:
----------------------------------------
Model String: models/gemini-2.5-flash
Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.

Model String: models/gemini-2.5-pro
Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro

Model String: models/gemini-2.5-flash-preview-tts
Description: Gemini 2.5 Flash Preview TTS

Model String: models/gemini-2.5-pro-preview-tts
Description: Gemini 2.5 Pro Preview TTS

Model String: models/gemma-4-26b-a4b-it
Description: Gemma 4 26B A4B IT

Model String: models/gemma-4-31b-it
Description: Gemma 4 31B IT

Model String: models/gemini-flash-latest
Description: Latest release of Gemini Flash

Model String: models/gemini-flash-lite-latest
Description: Latest release of Gemini Flash-Lite

Model String: models/gemini-pro-latest
Description: Latest release of Gemini Pro

Model String: models/gemini-2.5-flash-lite
Descrip

In [35]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0,
    google_api_key=os.environ.get("GOOGLE_API_KEY")
)

In [36]:
response=llm.invoke("What year is it?")
response

AIMessage(content="As an AI, I don't have access to a real-time clock, so I don't know the exact current year. However, my knowledge base includes information up to January 2025.", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-43173b2c-73ba-4768-a959-eb66857627ee-0')

In [37]:
prompt=hub.pull("hwchase17/react")

/home/anu/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [38]:
tools = [search_tool,get_weather_data]

In [39]:
agent=create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [40]:
agent_executor=AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

In [41]:
response=agent_executor.invoke({
    "input": (
        "What is the capital of India."
        "and what is the current weather there?"
    )
})



> Entering new AgentExecutor chain...
Thought: I need to find the capital of India and then check the current weather in that city.
Action: tavily_search_results_json
Action Input: capital of India[{'url': 'https://www.facebook.com/groups/1844186605882538/posts/3474689866165529', 'content': 'The capital of India is New Delhi . It is the seat of the Indian government and a city rich in history, culture, and modern development,'}, {'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_India', 'content': 'This is a list of locations which have served as capital cities in India. The current capital city is New Delhi, which replaced Calcutta in 1911. Contents.'}]Action: get_weather_data
Action Input: New DelhiCity: New Delhi
Temperature: 30°C
Weather: Smoky haze
Humidity: 44%
Final Answer: The capital of India is New Delhi. The current weather in New Delhi is 30°C with a smoky haze and 44% humidity.

> Finished chain.


In [28]:
print(response["output"])

The capital of India is New Delhi. The current weather there is smoky haze with a temperature of 30°C and a humidity level of 44%.
